In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
%pip install fast-langdetect

In [ ]:
from fast_langdetect import detect_language, detect

example2 = detect("Hello 世界 こんにちは")

print(example2)

In [ ]:
from fast_langdetect import detect

def clean_phomt_test_set(df):
    """
    Identifies rows where the Vietnamese ground truth is actually English.
    """
    def is_pass_through(text):
        if not isinstance(text, str) or len(text.strip()) < 5:
            return False

        result = detect(text)

        if (len(result) == 1) and (result[0]['lang'] == 'en'):
            return False
        else:
            return True

    # Apply the check to the Vietnamese column
    df['is_vietnamese'] = df['Ground_Truth'].apply(is_pass_through)

    # Filter the dataframe
    clean_df = df[~df['is_vietnamese']].copy()

    return clean_df

# Ground Truth

In [ ]:
results_ground_truth = pd.read_csv('/content/results_PhoMT_ground_truth.csv')
results_ground_truth.drop(columns=['Model_Translation'], inplace=True)

In [ ]:
results_ground_truth_not_clean = clean_phomt_test_set(results_ground_truth)
results_ground_truth_not_clean.shape

In [ ]:
results_ground_truth = results_ground_truth[results_ground_truth['is_vietnamese']]
results_ground_truth.shape

In [ ]:
# Get descriptive statistics for the COMET-22 score
comet_stats = results_ground_truth['COMET_Kiwi'].describe()
print("COMET-Kiwi Score Statistics:")
print(comet_stats)

In [ ]:
# Visualize the distribution
plt.figure(figsize=(10, 6))
sns.histplot(results_ground_truth['COMET_Kiwi'], kde=True, color='skyblue')
plt.title('Distribution of COMET-Kiwi Scores (Ground Truth)')
plt.xlabel('COMET-Kiwi Score')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=results_ground_truth['COMET_Kiwi'], color='lightgreen')
plt.title('Boxplot of COMET-22 Scores (Ground Truth)')
plt.xlabel('COMET-Kiwi Score')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
results_ground_truth[results_ground_truth['COMET_Kiwi'] > 0.8].shape

# Qwen3.5-0.8B

In [ ]:
results_qwen3_5_0_8b = pd.read_csv('/content/results_PhoMT_qwen3.5-0.8b.csv')

In [ ]:
results_qwen3_5_0_8b_not_clean = clean_phomt_test_set(results_qwen3_5_0_8b)
results_qwen3_5_0_8b_not_clean.shape

In [ ]:
results_qwen3_5_0_8b = results_qwen3_5_0_8b[results_qwen3_5_0_8b['is_vietnamese']]
results_qwen3_5_0_8b.shape

In [ ]:
# Get descriptive statistics for the COMET-22 score
comet_stats = results_qwen3_5_0_8b['COMET_Kiwi'].describe()
print("COMET-Kiwi Score Statistics:")
print(comet_stats)

In [ ]:
# Visualize the distribution
plt.figure(figsize=(10, 6))
sns.histplot(results_qwen3_5_0_8b['COMET_Kiwi'], kde=True, color='skyblue')
plt.title('Distribution of COMET-Kiwi Scores (Qwen3.5-0.8B)')
plt.xlabel('COMET-Kiwi Score')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=results_qwen3_5_0_8b['COMET_Kiwi'], color='lightgreen')
plt.title('Boxplot of COMET-Kiwi Scores (Qwen3.5-0.8B)')
plt.xlabel('COMET-Kiwi Score')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

gt_scores_df = results_ground_truth[['English', 'Ground_Truth', 'COMET_Kiwi']].rename(columns={'COMET_Kiwi': 'COMET_Kiwi_GT'})
qwen_scores_df = results_qwen3_5_0_8b[['English', 'Model_Translation', 'COMET_Kiwi']].rename(columns={'COMET_Kiwi': 'COMET_Kiwi_Qwen'})

# Merge the two dataframes on the 'English' column to ensure we compare scores for the same source sentences.
# Use an inner merge to only keep sentences present in both cleaned datasets.
comparison_df = pd.merge(gt_scores_df, qwen_scores_df, on='English', how='inner')

if comparison_df.empty:
    print("No common sentences found after cleaning and merging. Cannot create comparison matrix.")
else:
    # 4. Define the bins for COMET-Kiwi scores
    # Scores typically range from 0 to 1. Let's create bins from 0.0 to 1.0 with a step of 0.1.
    bins = np.arange(0.0, 1.01, 0.1) # Creates edges: [0.0, 0.1, ..., 1.0]

    # 5. Discretize the scores into bins
    # pd.cut with right=True means (a, b], include_lowest=True means the first bin is [min_score, bin_0].
    # This setup creates interval labels like [0.0, 0.1], (0.1, 0.2], ..., (0.9, 1.0].
    comparison_df['COMET_Kiwi_GT_Bin'] = pd.cut(comparison_df['COMET_Kiwi_GT'], bins=bins, include_lowest=True, right=True)
    comparison_df['COMET_Kiwi_Qwen_Bin'] = pd.cut(comparison_df['COMET_Kiwi_Qwen'], bins=bins, include_lowest=True, right=True)

    # 6. Create and Visualize the "Confusion Matrix"
    # Create the confusion matrix-like heatmap data using crosstab.
    confusion_matrix_data = pd.crosstab(
        comparison_df['COMET_Kiwi_GT_Bin'],
        comparison_df['COMET_Kiwi_Qwen_Bin'],
        dropna=False # Keep all defined bins even if they have no entries for better visualization.
    )

    # Ensure the order of bins is correct on the plot axes.
    # The categories of the pd.IntervalIndex from pd.cut are naturally ordered.
    ordered_bins = comparison_df['COMET_Kiwi_GT_Bin'].cat.categories
    confusion_matrix_data = confusion_matrix_data.reindex(index=ordered_bins, columns=ordered_bins, fill_value=0)

    # Plot the heatmap
    plt.figure(figsize=(14, 12)) # Adjusted figure size for better readability.
    sns.heatmap(confusion_matrix_data, annot=True, fmt='d', cmap='YlGnBu', linewidths=.5, cbar_kws={'label': 'Count of Translations'})
    plt.title('COMET-Kiwi Score Distribution Comparison: Ground Truth vs. Qwen3.5-0.8B')
    plt.xlabel('Qwen3.5-0.8B COMET-Kiwi Score Bins')
    plt.ylabel('Ground Truth COMET-Kiwi Score Bins')
    plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability.
    plt.yticks(rotation=0)
    plt.tight_layout() # Adjust layout to prevent labels from overlapping.
    plt.show()

In [ ]:
# Define the specific bins for filtering
gt_bin_lower = 0.8
gt_bin_upper = 0.9
qwen_bin_lower = 0.4
qwen_bin_upper = 0.5

# Create Interval objects for filtering, matching how pd.cut generates them
gt_interval = pd.Interval(left=gt_bin_lower, right=gt_bin_upper, closed='right')
qwen_interval = pd.Interval(left=qwen_bin_lower, right=qwen_bin_upper, closed='right')

# Filter the comparison_df for the specified bins
samples_in_bin = comparison_df[
    (comparison_df['COMET_Kiwi_GT_Bin'] == gt_interval) &
    (comparison_df['COMET_Kiwi_Qwen_Bin'] == qwen_interval)
]

print(f"Found {len(samples_in_bin)} samples where Ground Truth COMET-Kiwi is in {gt_interval} and Qwen3.5-0.8B COMET-Kiwi is in {qwen_interval}:\n")

# Display the relevant columns for these samples, including Ground Truth and Model Translation
# Limiting to first 20 samples if there are many to avoid overwhelming output
display(samples_in_bin[['English', 'Ground_Truth', 'Model_Translation', 'COMET_Kiwi_GT', 'COMET_Kiwi_Qwen']].head(20))

# Qwen3.5-0.8B LoRA fine-tuned

In [ ]:
results_qwen3_5_0_8b_lora_finetuned = pd.read_csv('/content/results_PhoMT_qwen3.5-0.8b_lora_fine_tuning.csv')

In [ ]:
results_qwen3_5_0_8b_lora_finetuned_not_clean = clean_phomt_test_set(results_qwen3_5_0_8b_lora_finetuned)
results_qwen3_5_0_8b_lora_finetuned_not_clean.shape

In [ ]:
results_qwen3_5_0_8b_lora_finetuned = results_qwen3_5_0_8b_lora_finetuned[results_qwen3_5_0_8b_lora_finetuned['is_vietnamese']]
results_qwen3_5_0_8b_lora_finetuned.shape

In [ ]:
# Get descriptive statistics for the COMET-22 score
comet_stats = results_qwen3_5_0_8b_lora_finetuned['COMET_Kiwi'].describe()
print("COMET-Kiwi Score Statistics:")
print(comet_stats)

In [ ]:
# Visualize the distribution
plt.figure(figsize=(10, 6))
sns.histplot(results_qwen3_5_0_8b_lora_finetuned['COMET_Kiwi'], kde=True, color='skyblue')
plt.title('Distribution of COMET-Kiwi Scores (Qwen3.5-0.8B Fine-tuned)')
plt.xlabel('COMET-Kiwi Score')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=results_qwen3_5_0_8b_lora_finetuned['COMET_Kiwi'], color='lightgreen')
plt.title('Boxplot of COMET-Kiwi Scores (Qwen3.5-0.8B Fine-tuned)')
plt.xlabel('COMET-Kiwi Score')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

gt_scores_df = results_ground_truth[['English', 'Ground_Truth', 'COMET_Kiwi']].rename(columns={'COMET_Kiwi': 'COMET_Kiwi_GT'})
qwen_scores_df = results_qwen3_5_0_8b_lora_finetuned[['English', 'Model_Translation', 'COMET_Kiwi']].rename(columns={'COMET_Kiwi': 'COMET_Kiwi_Qwen'})

# Merge the two dataframes on the 'English' column to ensure we compare scores for the same source sentences.
# Use an inner merge to only keep sentences present in both cleaned datasets.
comparison_df = pd.merge(gt_scores_df, qwen_scores_df, on='English', how='inner')

if comparison_df.empty:
    print("No common sentences found after cleaning and merging. Cannot create comparison matrix.")
else:
    # 4. Define the bins for COMET-Kiwi scores
    # Scores typically range from 0 to 1. Let's create bins from 0.0 to 1.0 with a step of 0.1.
    bins = np.arange(0.0, 1.01, 0.1) # Creates edges: [0.0, 0.1, ..., 1.0]

    # 5. Discretize the scores into bins
    # pd.cut with right=True means (a, b], include_lowest=True means the first bin is [min_score, bin_0].
    # This setup creates interval labels like [0.0, 0.1], (0.1, 0.2], ..., (0.9, 1.0].
    comparison_df['COMET_Kiwi_GT_Bin'] = pd.cut(comparison_df['COMET_Kiwi_GT'], bins=bins, include_lowest=True, right=True)
    comparison_df['COMET_Kiwi_Qwen_Bin'] = pd.cut(comparison_df['COMET_Kiwi_Qwen'], bins=bins, include_lowest=True, right=True)

    # 6. Create and Visualize the "Confusion Matrix"
    # Create the confusion matrix-like heatmap data using crosstab.
    confusion_matrix_data = pd.crosstab(
        comparison_df['COMET_Kiwi_GT_Bin'],
        comparison_df['COMET_Kiwi_Qwen_Bin'],
        dropna=False # Keep all defined bins even if they have no entries for better visualization.
    )

    # Ensure the order of bins is correct on the plot axes.
    # The categories of the pd.IntervalIndex from pd.cut are naturally ordered.
    ordered_bins = comparison_df['COMET_Kiwi_GT_Bin'].cat.categories
    confusion_matrix_data = confusion_matrix_data.reindex(index=ordered_bins, columns=ordered_bins, fill_value=0)

    # Plot the heatmap
    plt.figure(figsize=(14, 12)) # Adjusted figure size for better readability.
    sns.heatmap(confusion_matrix_data, annot=True, fmt='d', cmap='YlGnBu', linewidths=.5, cbar_kws={'label': 'Count of Translations'})
    plt.title('COMET-Kiwi Score Distribution Comparison: Ground Truth vs. Qwen3.5-0.8B')
    plt.xlabel('Qwen3.5-0.8B COMET-Kiwi Score Bins')
    plt.ylabel('Ground Truth COMET-Kiwi Score Bins')
    plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability.
    plt.yticks(rotation=0)
    plt.tight_layout() # Adjust layout to prevent labels from overlapping.
    plt.show()

In [ ]:
# Define the specific bins for filtering
gt_bin_lower = 0.8
gt_bin_upper = 0.9
qwen_bin_lower = 0.4
qwen_bin_upper = 0.5

# Create Interval objects for filtering, matching how pd.cut generates them
gt_interval = pd.Interval(left=gt_bin_lower, right=gt_bin_upper, closed='right')
qwen_interval = pd.Interval(left=qwen_bin_lower, right=qwen_bin_upper, closed='right')

# Filter the comparison_df for the specified bins
samples_in_bin = comparison_df[
    (comparison_df['COMET_Kiwi_GT_Bin'] == gt_interval) &
    (comparison_df['COMET_Kiwi_Qwen_Bin'] == qwen_interval)
]

print(f"Found {len(samples_in_bin)} samples where Ground Truth COMET-Kiwi is in {gt_interval} and Qwen3.5-0.8B COMET-Kiwi is in {qwen_interval}:\n")

# Display the relevant columns for these samples, including Ground Truth and Model Translation
# Limiting to first 20 samples if there are many to avoid overwhelming output
display(samples_in_bin[['English', 'Ground_Truth', 'Model_Translation', 'COMET_Kiwi_GT', 'COMET_Kiwi_Qwen']].head(20))

# Qwen3.5-2B

In [ ]:
results_qwen3_5_2b = pd.read_csv('/content/results_PhoMT_qwen3.5-2b.csv')

In [ ]:
# Get descriptive statistics for the COMET-22 score
comet_stats = results_qwen3_5_2b['COMET_Kiwi'].describe()
print("COMET-Kiwi Score Statistics:")
print(comet_stats)

In [ ]:
# Visualize the distribution
plt.figure(figsize=(10, 6))
sns.histplot(results_qwen3_5_2b['COMET_Kiwi'], kde=True, color='skyblue')
plt.title('Distribution of COMET-Kiwi Scores (Qwen3.5-2B)')
plt.xlabel('COMET-Kiwi Score')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=results_qwen3_5_2b['COMET_Kiwi'], color='lightgreen')
plt.title('Boxplot of COMET-Kiwi Scores (Qwen3.5-2B)')
plt.xlabel('COMET-Kiwi Score')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# TranslateGemma-4B-IT

In [ ]:
results_translategemma = pd.read_csv('/content/results_PhoMT_translategemma-4b-it.csv')

In [ ]:
# Get descriptive statistics for the COMET-22 score
comet_stats = results_translategemma['COMET_Kiwi'].describe()
print("COMET-Kiwi Score Statistics:")
print(comet_stats)

In [ ]:
# Visualize the distribution
plt.figure(figsize=(10, 6))
sns.histplot(results_translategemma['COMET_Kiwi'], kde=True, color='skyblue')
plt.title('Distribution of COMET-Kiwi Scores (TranslateGemma)')
plt.xlabel('COMET-Kiwi Score')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=results_translategemma['COMET_Kiwi'], color='lightgreen')
plt.title('Boxplot of COMET-Kiwi Scores (TranslateGemma)')
plt.xlabel('COMET-Kiwi Score')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# VinAI Translate

In [ ]:
results_vinai_translate = pd.read_csv('/content/results_PhoMT_vinai_translate.csv')

In [ ]:
# Get descriptive statistics for the COMET-22 score
comet_stats = results_vinai_translate['COMET_Kiwi'].describe()
print("COMET-Kiwi Score Statistics:")
print(comet_stats)

In [ ]:
# Visualize the distribution
plt.figure(figsize=(10, 6))
sns.histplot(results_vinai_translate['COMET_Kiwi'], kde=True, color='skyblue')
plt.title('Distribution of COMET-Kiwi Scores (VinAI Translate)')
plt.xlabel('COMET-Kiwi Score')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=results_vinai_translate['COMET_Kiwi'], color='lightgreen')
plt.title('Boxplot of COMET-22 Scores (VinAI Translate)')
plt.xlabel('COMET-Kiwi Score')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# Llama3.2-1B-IT

In [ ]:
results_llama3_2_1b_it = pd.read_csv('/content/results_PhoMT_llama3.2-1b-it.csv')

In [ ]:
# Get descriptive statistics for the COMET-22 score
comet_stats = results_llama3_2_1b_it['COMET_Kiwi'].describe()
print("COMET-Kiwi Score Statistics:")
print(comet_stats)

In [ ]:
# Visualize the distribution
plt.figure(figsize=(10, 6))
sns.histplot(results_llama3_2_1b_it['COMET_Kiwi'], kde=True, color='skyblue')
plt.title('Distribution of COMET-Kiwi Scores (Llama3.2-1B-IT)')
plt.xlabel('COMET-Kiwi Score')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=results_llama3_2_1b_it['COMET_Kiwi'], color='lightgreen')
plt.title('Boxplot of COMET-Kiwi Scores (Llama 3.2-1B-IT)')
plt.xlabel('COMET-Kiwi Score')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# Find outlier

In [ ]:
def get_outliers(df, column='COMET_Kiwi'):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers

In [ ]:
outliers_qwen3_5_0_8b = get_outliers(results_qwen3_5_0_8b)

In [ ]:
outliers_qwen3_5_0_8b.shape

In [ ]:
outliers_qwen3_5_0_8b.to_csv('outliers_PhoMT_qwen3_5_0_8b_COMET-Kiwi.csv', index=False)
print("Outliers saved")

In [ ]:
outliers_qwen3_5_2b = get_outliers(results_qwen3_5_2b)

In [ ]:
outliers_qwen3_5_2b.shape

In [ ]:
outliers_qwen3_5_2b.to_csv('outliers_PhoMT_qwen3_5_2b_COMET-Kiwi.csv', index=False)
print("Outliers saved")

In [ ]:
outliers_ground_truth = get_outliers(results_ground_truth)

In [ ]:
outliers_ground_truth.shape

In [ ]:
outliers_ground_truth.to_csv('outliers_PhoMT_ground_truth_COMET-Kiwi.csv', index=False)
print("Outliers saved")

In [ ]:
output_filename = 'qwen3_5_0_8b_outliers.txt'

with open(output_filename, 'w', encoding='utf-8') as f:
    for index, row in outliers_qwen3_5_0_8b.iterrows():
        f.write(f"--- Outlier {index} ---\n")
        f.write(f"English: {row['English']}\n")
        f.write(f"Ground Truth: {row['Ground_Truth']}\n")
        f.write(f"Model Translation: {row['Model_Translation']}\n")
        f.write(f"COMET-Kiwi Score: {row['COMET_Kiwi']}\n")
        f.write("\n")

print(f"Outlier details written to {output_filename}")